In [1]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------------------------------------
# SELECIONE AS PASTAS DE RESULTADO DESEJADAS (result_XXXX)
# Deixe a lista vazia [] para carregar TODAS as pastas encontradas.
# Exemplo: RESULT_DIRS = ['result_0005', 'result_0006']
# -----------------------------------------------------------
RESULT_DIRS = ['result_0006']  # <-- ALTERE AQUI
# -----------------------------------------------------------

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
RESULTS_BASE = PROJECT_ROOT / 'project' / 'results'
BEST_OBJ_CSV = PROJECT_ROOT / 'best_solutions' / 'best_objectives.csv'

# --- Carrega melhores objetivos ---
df_best = pd.read_csv(BEST_OBJ_CSV)
df_best['instance'] = df_best['instance'].str.replace('.txt', '', regex=False)

# --- Carrega resultados ---
def carregar_resultados(results_base):
    registros = []
    if RESULT_DIRS:
        result_dirs = sorted([Path(results_base) / d for d in RESULT_DIRS])
    else:
        result_dirs = sorted(Path(results_base).glob('result_*'))
    if not result_dirs:
        print('[AVISO] Nenhuma pasta result_XXXX encontrada em:', results_base)
        return pd.DataFrame()
    for result_dir in result_dirs:
        config_path = result_dir / 'config.json'
        dataset_map = {}
        if config_path.exists():
            with open(config_path) as f:
                cfg = json.load(f)
            for ds, instances in cfg.get('datasets', {}).items():
                for inst in instances:
                    inst_stem = inst.replace('.txt', '')
                    dataset_map[inst_stem] = ds
        csv_files = sorted(result_dir.glob('instance_*.csv'))
        for csv_file in csv_files:
            instance_stem = csv_file.stem
            dataset = dataset_map.get(instance_stem, 'desconhecido')
            try:
                df_inst = pd.read_csv(csv_file)
                df_inst['dataset']    = dataset
                df_inst['instance']   = instance_stem
                df_inst['result_dir'] = result_dir.name
                registros.append(df_inst)
            except Exception as e:
                print('[ERRO] Falha ao ler', csv_file, ':', e)
    if not registros:
        print('[AVISO] Nenhum CSV de instancia encontrado.')
        return pd.DataFrame()
    return pd.concat(registros, ignore_index=True)

df_raw = carregar_resultados(RESULTS_BASE)
print('Total de linhas carregadas:', len(df_raw))

# --- Filtra feasible ---
if df_raw.empty:
    df_ok = pd.DataFrame()
else:
    df_ok = df_raw[df_raw['status'] == 'feasible'].copy()
    print('Execucoes com status feasible:', len(df_ok), 'de', len(df_raw))
    df_ok['objective'] = pd.to_numeric(df_ok['objective'], errors='coerce')
    df_ok['exec_time'] = pd.to_numeric(df_ok['exec_time'], errors='coerce')

# --- Calcula gap ---
if not df_ok.empty:
    df_ok = df_ok.merge(
        df_best[['dataset', 'instance', 'best_objective']],
        on=['dataset', 'instance'],
        how='left'
    )
    n_sem_best = df_ok['best_objective'].isna().sum()
    if n_sem_best > 0:
        print(f'[AVISO] {n_sem_best} linhas sem melhor objetivo conhecido.')
    df_ok['gap'] = (
        (df_ok['best_objective'] - df_ok['objective']) / df_ok['best_objective']
    )
    print('Gap calculado com sucesso.')

# --- Tabela por Dataset + Algoritmo ---
def formata(media, dp, casas=4):
    fmt = '{:.' + str(casas) + 'f}'
    return fmt.format(media) + ' +- ' + fmt.format(dp)

if df_ok.empty or 'gap' not in df_ok.columns:
    print('[INFO] Nenhum dado disponivel para gerar a tabela.')
else:
    df_valido = df_ok.dropna(subset=['gap', 'exec_time'])
    grupos = df_valido.groupby(['dataset', 'algo_id'])
    tabela = grupos.agg(
        gap_media   = ('gap',       'mean'),
        gap_dp      = ('gap',       'std'),
        tempo_media = ('exec_time', 'mean'),
        tempo_dp    = ('exec_time', 'std'),
        n_runs      = ('run_id',    'count'),
    ).reset_index()
    tabela['gap_dp']   = tabela['gap_dp'].fillna(0.0)
    tabela['tempo_dp'] = tabela['tempo_dp'].fillna(0.0)
    tabela = tabela.sort_values('gap_media', ascending=True)
    tabela['Gap media + DP'] = tabela.apply(lambda r: formata(r['gap_media'],   r['gap_dp']),   axis=1)
    tabela['Tempo media + DP']   = tabela.apply(lambda r: formata(r['tempo_media'], r['tempo_dp']), axis=1)
    tabela_exibir = tabela[['dataset', 'algo_id', 'n_runs', 'Gap media + DP', 'Tempo media + DP']].rename(columns={
        'dataset':  'Dataset',
        'algo_id':  'Algoritmo',
        'n_runs':   'Total Runs',
    })

    display(tabela_exibir.reset_index(drop=True))

# --- Exporta CSV ---
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

if not df_ok.empty and 'gap' in df_ok.columns:
    path = OUTPUT_DIR / 'tabela_por_dataset.csv'
    tabela_exibir.to_csv(path, index=False)
    print('Tabela salva em:', path)
else:
    print('[INFO] Nada exportado - nenhum dado disponivel.')

Total de linhas carregadas: 5700
Execucoes com status feasible: 5700 de 5700
Gap calculado com sucesso.


,Dataset,Algoritmo,Total Runs,Gap media + DP,Tempo media + DP
0,a,simple_desc_exact,100,0.6916 +- 0.2731,0.0756 +- 0.0939
1,a,simple_desc_multi,100,0.6945 +- 0.2707,0.3175 +- 0.4938
2,a,simple_desc_simple,100,0.7047 +- 0.2678,1.0949 +- 2.1202
3,a,simple_diff_exact_bigger,100,0.7127 +- 0.2884,0.0987 +- 0.1214
4,a,simple_diff_weighted_exact_bigger,100,0.7127 +- 0.2884,0.0949 +- 0.1074
5,a,simple_diff_multi_bigger,100,0.7159 +- 0.2872,0.3564 +- 0.5325
6,a,simple_diff_weighted_multi_bigger,100,0.7159 +- 0.2872,0.3555 +- 0.5690
7,a,simple_diff_exact_most_shared,100,0.7195 +- 0.2830,0.0994 +- 0.1166
8,a,simple_diff_weighted_exact_most_shared,100,0.7195 +- 0.2830,0.0978 +- 0.1017
9,a,simple_similar_exact_bigger,100,0.7227 +- 0.2603,0.0962 +- 0.1044


Tabela salva em: /home/vinicius/Documents/CEFET/TCC/pfc2/notebooks/output/tabela_por_dataset.csv
